# AGRI-INTELLIGENCE | PLATFORM NOTEBOOK V1
**ArqTech Labs · Multimodal Agricultural Intelligence**

Este notebook é a interface de orquestração no Google Colab. A lógica reutilizável permanece em `src/agri_intelligence`; dados, estado e artefatos persistem no Google Drive.

**Milestone atual:** bootstrap persistente e recuperação verificável das fontes da Phase 2.5. Nenhum treinamento ou inferência é executado nesta versão inicial.

Branch de desenvolvimento: `feat/platform-notebook-v1`.

## 0. Regras de execução

- Não instalar dependências automaticamente.
- Não baixar pesos nem iniciar treinamento/inferência nesta etapa.
- Não carregar ortomosaicos completos na RAM; usar leitura por janelas.
- Recuperar arquivos somente quando ausentes ou com checksum divergente.
- Preservar arquivos inválidos como backup antes de substituí-los.
- Interromper em checksum divergente, fonte inesperada ou branch incorreta.
- Cada etapa registra configuração, commit, artefatos e resultado.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import os, sys, json, hashlib, subprocess, platform

PROJECT_NAME = "Agri-Intelligence"
REPOSITORY = "edu-moraess/agri-intelligence"
REPOSITORY_URL = f"https://github.com/{REPOSITORY}.git"
DEVELOPMENT_BRANCH = "feat/platform-notebook-v1"
ZENODO_RECORD_ID = "15235602"
ZENODO_API = f"https://zenodo.org/api/records/{ZENODO_RECORD_ID}"

DRIVE_ROOT = Path("/content/drive/MyDrive/ArqTechLabs/Agri-Intelligence")
REPO_ROOT = Path("/content/agri-intelligence-platform-v1")
SOURCE_ROOT = DRIVE_ROOT / "data/raw/zenodo_15235602"
INTERIM_ROOT = DRIVE_ROOT / "data/interim/phase_2_5"
PROCESSED_ROOT = DRIVE_ROOT / "data/processed/phase_2_5"
STATE_ROOT = DRIVE_ROOT / "state"
ARTIFACT_ROOT = DRIVE_ROOT / "artifacts"
LOG_ROOT = DRIVE_ROOT / "logs"

EXPECTED_RASTER_MD5 = {
    "orthomosaic_1.tif": "062a5c9028fff72f3fc276917e0a5af1",
    "orthomosaic_2.tif": "c5c478aa73a22f11d52eb8176c4c8b1a",
    "orthomosaic_3.tif": "a800e5be8ec95f4a61d2c24a0b4e4a74",
}
EXPECTED_RASTER_SHA256 = {
    "orthomosaic_1.tif": "d4317d8eb402a15d33e084e131de26bd6340a14756148b4e6011a912f3695a2c",
    "orthomosaic_2.tif": "c5a81cfab297866a82633130f161ac3a31524956a17c9e94e98f8e6ac3583feb",
    "orthomosaic_3.tif": "86abb6bb4dca079aa07de0520e376e34fcaf21efaba93cfd587c520a4008948d",
}
EXPECTED_GT_COUNTS = {"gt_1": 61251, "gt_2": 30547, "gt_3": 55374}

print(f"{PROJECT_NAME} | PLATFORM V1")
print("Branch alvo:", DEVELOPMENT_BRANCH)
print("Drive:", DRIVE_ROOT)
print("Fonte:", ZENODO_API)

## 1. Montagem do armazenamento persistente

Os arquivos brutos e resultados ficam no Drive, não no disco efêmero de `/content`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

for path in [DRIVE_ROOT, SOURCE_ROOT, INTERIM_ROOT, PROCESSED_ROOT,
             STATE_ROOT, ARTIFACT_ROOT, ARTIFACT_ROOT / "manifests", LOG_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

print("PERSISTENT STORAGE: READY")
print("Source root:", SOURCE_ROOT)
print("State root:", STATE_ROOT)

## 2. Repositório e identidade da execução

Em uma sessão nova, o notebook clona a branch de desenvolvimento. Se o repositório já existir, não troca de branch nem sobrescreve alterações locais.

In [ ]:
def run_cmd(args, cwd=None):
    result = subprocess.run(args, cwd=cwd, text=True, capture_output=True)
    if result.returncode != 0:
        raise RuntimeError(result.stderr.strip() or result.stdout.strip())
    return result.stdout.strip()

if not (REPO_ROOT / ".git").exists():
    if REPO_ROOT.exists() and any(REPO_ROOT.iterdir()):
        raise RuntimeError(f"Diretório existe e não é um clone Git limpo: {REPO_ROOT}")
    run_cmd(["git", "clone", "--branch", DEVELOPMENT_BRANCH,
             "--single-branch", REPOSITORY_URL, str(REPO_ROOT)])

current_branch = run_cmd(["git", "branch", "--show-current"], cwd=REPO_ROOT)
current_commit = run_cmd(["git", "rev-parse", "HEAD"], cwd=REPO_ROOT)
working_tree = run_cmd(["git", "status", "--porcelain"], cwd=REPO_ROOT)

if current_branch != DEVELOPMENT_BRANCH:
    raise RuntimeError(
        f"Branch incorreta: {current_branch}. Esperada: {DEVELOPMENT_BRANCH}. "
        "Não foi feito checkout automático."
    )

sys.path.insert(0, str(REPO_ROOT / "src"))
print("BRANCH:", current_branch)
print("HEAD:", current_commit)
print("WORKTREE:", "CLEAN" if not working_tree else "MODIFIED")
if working_tree:
    print(working_tree)

### 2.1. Recuperar o checkpoint local da Phase 2.5

O bundle criado anteriormente no Drive pode conter o protocolo e os módulos L2 que não foram publicados no GitHub. Esta célula verifica e importa a referência para o clone novo **sem trocar de branch, fazer merge ou alterar a árvore de trabalho**.

In [ ]:
BUNDLE_PATH = Path("/content/drive/MyDrive/agri_phase25/phase25_protocol.bundle")
BUNDLE_REF = "refs/heads/feat/phase-2-5-first-real-experiment"

if not BUNDLE_PATH.is_file():
    print("PHASE 2.5 BUNDLE: NOT FOUND")
    print("A continuidade poderá ser reconstruída a partir da base remota, sem presumir que L2 esteja presente.")
else:
    verification = subprocess.run(
        ["git", "bundle", "verify", str(BUNDLE_PATH)],
        cwd=REPO_ROOT, text=True, capture_output=True
    )
    if verification.returncode != 0:
        raise RuntimeError("Bundle Phase 2.5 inválido: " + verification.stderr.strip())

    refs_output = run_cmd(["git", "bundle", "list-heads", str(BUNDLE_PATH)], cwd=REPO_ROOT)
    bundle_refs = [line.split()[-1] for line in refs_output.splitlines() if line.strip()]
    if BUNDLE_REF not in bundle_refs:
        raise RuntimeError(f"Referência esperada não encontrada no bundle. Refs: {bundle_refs}")

    run_cmd([
        "git", "fetch", str(BUNDLE_PATH),
        f"{BUNDLE_REF}:refs/remotes/backup/phase25"
    ], cwd=REPO_ROOT)
    bundle_commit = run_cmd(
        ["git", "rev-parse", "refs/remotes/backup/phase25"], cwd=REPO_ROOT
    )
    print("PHASE 2.5 BUNDLE: VERIFIED")
    print("Imported ref: refs/remotes/backup/phase25")
    print("Bundle commit:", bundle_commit)
    print("Current branch unchanged:", current_branch)
    bundle_state = {
        "bundle_path": str(BUNDLE_PATH),
        "source_ref": BUNDLE_REF,
        "imported_ref": "refs/remotes/backup/phase25",
        "commit": bundle_commit,
        "verified_at_utc": datetime.now(timezone.utc).isoformat(),
    }
    bundle_state_path = STATE_ROOT / "phase_2_5_bundle_recovery.json"
    bundle_state_temp = bundle_state_path.with_name(bundle_state_path.name + ".part")
    bundle_state_temp.write_text(json.dumps(bundle_state, indent=2), encoding="utf-8")
    bundle_state_temp.replace(bundle_state_path)

## 3. Compatibilidade do ambiente

Esta etapa apenas detecta dependências. Não instala nem atualiza pacotes.

In [ ]:
import importlib.util
import sys

OPTIONAL_MODULES = ["numpy", "pandas", "rasterio", "geopandas", "shapely", "requests", "PIL"]
availability = {name: importlib.util.find_spec(name) is not None for name in OPTIONAL_MODULES}

runtime = {
    "python": sys.version,
    "platform": platform.platform(),
    "modules": availability,
    "branch": current_branch,
    "commit": current_commit,
    "checked_at_utc": datetime.now(timezone.utc).isoformat(),
}

STATE_ROOT.mkdir(parents=True, exist_ok=True)
(STATE_ROOT / "runtime_environment.json").write_text(
    json.dumps(runtime, indent=2), encoding="utf-8"
)

for name, available in availability.items():
    print(f"{'AVAILABLE' if available else 'MISSING'} | {name}")
print("Python:", sys.version.split()[0])
print("Nenhuma instalação automática foi executada.")

## 4. Recuperação persistente das fontes — Zenodo 15235602

O conjunto contém três GeoTIFFs e três famílias Shapefile. Cada Shapefile depende de arquivos auxiliares (`.shp`, `.shx`, `.dbf`, `.prj`, `.cpg`; e metadados `.qmd` quando publicados). O processo abaixo seleciona somente os ortomosaicos e GT. Não utiliza `HC_bboxes_*` nem `tiles_*` como ground truth.

A recuperação é retomável: arquivos válidos são preservados; downloads são gravados como `.part`, verificados por tamanho e MD5 do registro e só então renomeados. SHA-256 é calculado para a proveniência local.

In [ ]:
import requests, shutil, time

def digest_file(path, algorithm="sha256", chunk_size=8 * 1024 * 1024):
    h = hashlib.new(algorithm)
    with open(path, "rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

def atomic_json(path, payload):
    path = Path(path)
    temp = path.with_name(path.name + ".part")
    temp.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
    temp.replace(path)

response = requests.get(ZENODO_API, timeout=(30, 120))
response.raise_for_status()
record = response.json()
if str(record.get("id")) != ZENODO_RECORD_ID:
    raise RuntimeError("O registro Zenodo retornado não corresponde ao ID solicitado.")

raw_files = record.get("files", [])
if isinstance(raw_files, dict):
    if isinstance(raw_files.get("entries"), list):
        raw_files = raw_files["entries"]
    else:
        raw_files = list(raw_files.values())
file_index = {item.get("key") or item.get("filename"): item for item in raw_files}

core_names = [f"orthomosaic_{i}.tif" for i in (1, 2, 3)]
gt_extensions = [".shp", ".shx", ".dbf", ".prj", ".cpg"]
gt_names = [f"gt_{i}{ext}" for i in (1, 2, 3) for ext in gt_extensions]
optional_names = [f"gt_{i}.qmd" for i in (1, 2, 3)]
required_names = core_names + gt_names
missing_metadata = sorted(set(required_names) - set(file_index))
if missing_metadata:
    raise RuntimeError(f"Arquivos obrigatórios ausentes no registro: {missing_metadata}")

metadata = record.get("metadata", {})
license_metadata = metadata.get("license") or metadata.get("rights")
print("Registro:", record.get("doi") or f"Zenodo {ZENODO_RECORD_ID}")
print("Título:", metadata.get("title", "não informado"))
print("Licença declarada na API:", license_metadata)
if not license_metadata:
    print("ATENÇÃO: API não expôs licença estruturada; manter ressalva documental.")

selected_names = required_names + [n for n in optional_names if n in file_index]
recovery_inventory = []

for index, name in enumerate(selected_names, start=1):
    info = file_index[name]
    expected_size = int(info.get("size", -1))
    checksum = info.get("checksum", "")
    if not checksum.startswith("md5:") or expected_size < 0:
        raise RuntimeError(f"Metadados de tamanho/checksum inválidos para {name}: {info}")
    expected_md5 = checksum.split(":", 1)[1].lower()
    target = SOURCE_ROOT / name
    target.parent.mkdir(parents=True, exist_ok=True)

    valid_existing = (
        target.is_file()
        and target.stat().st_size == expected_size
        and digest_file(target, "md5") == expected_md5
    )
    if valid_existing:
        print(f"[{index}/{len(selected_names)}] VALIDADO | {name}")
    else:
        if target.exists():
            stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
            backup = target.with_name(target.name + f".invalid-{stamp}")
            target.replace(backup)
            print(f"BACKUP DE ARQUIVO INVÁLIDO | {backup.name}")

        links = info.get("links", {})
        download_url = links.get("download") or links.get("content") or links.get("self")
        if not download_url:
            raise RuntimeError(f"URL de download ausente para {name}")

        temp = target.with_name(target.name + ".part")
        md5 = hashlib.md5()
        sha256 = hashlib.sha256()
        size = 0
        print(f"[{index}/{len(selected_names)}] BAIXANDO | {name} | {expected_size / 1e9:.3f} GB")
        with requests.get(download_url, stream=True, timeout=(30, 240)) as download:
            download.raise_for_status()
            with open(temp, "wb") as output:
                for chunk in download.iter_content(chunk_size=8 * 1024 * 1024):
                    if not chunk:
                        continue
                    output.write(chunk)
                    md5.update(chunk)
                    sha256.update(chunk)
                    size += len(chunk)
        if size != expected_size or md5.hexdigest().lower() != expected_md5:
            temp.unlink(missing_ok=True)
            raise RuntimeError(
                f"Integridade divergente em {name}: size={size}/{expected_size}, "
                f"md5={md5.hexdigest()}/{expected_md5}"
            )
        temp.replace(target)

    local_sha256 = digest_file(target, "sha256")
    if name in EXPECTED_RASTER_SHA256 and local_sha256 != EXPECTED_RASTER_SHA256[name]:
        raise RuntimeError(f"SHA-256 histórico divergente para {name}: {local_sha256}")
    recovery_inventory.append({
        "name": name,
        "relative_path": name,
        "size_bytes": target.stat().st_size,
        "zenodo_md5": expected_md5,
        "local_sha256": local_sha256,
        "download_url": download_url if not valid_existing else None,
        "status": "VERIFIED",
    })

source_manifest = {
    "dataset_id": "maize_seedling_orthomosaics_zenodo_15235602",
    "record_id": ZENODO_RECORD_ID,
    "doi": record.get("doi", "10.5281/zenodo.15235602"),
    "record_version": metadata.get("version"),
    "title": metadata.get("title"),
    "license_metadata": license_metadata,
    "license_caveat": "Preservar a ressalva: direitos de terceiros do depositante não foram verificados independentemente.",
    "retrieved_at_utc": datetime.now(timezone.utc).isoformat(),
    "source_root": str(SOURCE_ROOT),
    "files": recovery_inventory,
    "result": "PASS",
}
atomic_json(ARTIFACT_ROOT / "manifests" / "phase_2_5_source_manifest.json", source_manifest)
print("\nSOURCE RECOVERY: PASS")
print("Arquivos recuperados/validados:", len(recovery_inventory))
print("Dados persistidos em:", SOURCE_ROOT)
print("Manifesto:", ARTIFACT_ROOT / "manifests" / "phase_2_5_source_manifest.json")

## 5. Caracterização do ambiente e fontes

Com os arquivos persistidos e verificados, a próxima etapa será reabrir os GeoTIFFs por janela, confirmar dimensões/CRS/transform/bandas e ler os GT Shapefiles. Esses dados serão ligados ao manifesto existente. A implementação espacial e a materialização do dataset ficam em módulos testáveis no repositório, não neste notebook.

## 6. Roadmap operacional

| Marco | Objetivo | Estado |
|---|---|---|
| Bootstrap | Drive, branch, ambiente e identidade de execução | Implementado neste notebook |
| Recovery | Recuperar fontes e validar checksums | Implementado neste notebook |
| L3 | Seleção espacial, deduplicação e splits | Próximo |
| L4 | Materialização RGB/YOLO e conservação de GT | Pendente |
| L5 | Manifesto, integridade e leakage espacial | Pendente |
| L6 | Smoke test e treinamento controlado | Bloqueado pelos gates de dados |
| L7 | Avaliação e relatório reproduzível | Pendente |

Depois da primeira prova de percepção, a plataforma evolui para Sensor Intelligence, Environmental Intelligence, Multimodal Fusion, Temporal Intelligence, Agricultural Context, Predictive/Anomaly Intelligence e Decision Intelligence.

## 7. Limites deste milestone

- Este notebook ainda não declara L3, L4 ou L5 aprovadas.
- Nenhum treinamento ou inferência foi executado.
- A recuperação do dataset não equivale à aprovação jurídica de todos os direitos de uso.
- Os resultados agronômicos exigem validação independente em campo.
- O próximo incremento deverá reutilizar os contratos e módulos já existentes, sem duplicar Data Engine, percepção ou avaliação.